In [22]:
from datetime import datetime, timedelta

yesterday = datetime.now() - timedelta(days=1)
date_str = yesterday.strftime("%Y-%m-%d")  # 예: 2026-03-18

base_path = f"/home/axtft/projects/axtft/logs/report/interval/{date_str}"

In [23]:
import glob

json_files = glob.glob(f"{base_path}/*.json")

In [26]:
import json

data_list = []

for file_path in json_files:
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            data_list.append(data)

    except Exception as e:
        print(f"읽기 실패: {file_path} → {e}")

In [27]:
from datetime import datetime

def parse_time(t: str):
    return datetime.strptime(t, "%H:%M:%S")

filtered = []

for item in data_list:
    try:
        level = item.get("level")
        time_window = item.get("timeWindow", {})
        start_time = time_window.get("start")

        # 필수값 체크
        if not start_time:
            continue

        # low 제외
        if level == "low":
            continue

        filtered.append({
            "level": level,
            "summary": item.get("summary"),
            "start": start_time,
            "end": time_window.get("end"),
            "events": item.get("events", [])
        })

    except Exception as e:
        print("파싱 실패:", e)

In [29]:
filtered_sorted = sorted(
    filtered,
    key=lambda x: parse_time(x["start"])
)

In [30]:
def build_llm_input(data):
    lines = []

    for item in data:
        line = (
            f"[{item['start']} ~ {item['end']}] "
            f"level={item['level']} | "
            f"{item['summary']}"
        )
        lines.append(line)

        # 이벤트도 포함 (있으면)
        for ev in item.get("events", []):
            ev_line = (
                f"  - ({ev.get('start_time')} ~ {ev.get('end_time')}) "
                f"{ev.get('summary')}"
            )
            lines.append(ev_line)

    return "\n".join(lines)

In [41]:
llm_input_text = build_llm_input(filtered_sorted)

In [42]:
user_prompt = f"""
Analyze the following daily events and generate a report.

{llm_input_text}
"""

In [ ]:
with open("/home/axtft/projects/axtft/app/prompts/daily_report_system_prompt.txt", "r", encoding="utf-8") as f:
    system_prompt = f.read()

'You are a strict, deterministic daily service-report analyst.\n\nYour task is to generate a concise daily operational report from time-ordered event summaries.\n\n========================\nSEVERITY POLICY (STRICT)\n========================\n\nAllowed levels:\n- low\n- medium\n- high\n\n"critical" MUST NEVER be used.\n\nDefinition:\n\nLOW:\n- Normal or near-normal state\n- No meaningful performance or service impact\n\nMEDIUM:\n- Noticeable performance degradation\n- latency spikes, resource pressure\n- but service is still available\n\nHIGH:\n- Real service impact\n- Examples:\n  - service_status = 0\n  - throughput = 0\n  - request failure or outage\n\nIMPORTANT:\n- Use "high" ONLY for real service disruption\n- Repeated issues without outage → still "medium"\n\n========================\nOUTPUT RULES\n========================\n\n- Output ONLY a single valid JSON object\n- No markdown or explanation\n- All descriptions in Korean\n- Keep output concise (1~2 sentences per field)\n- Avoi

In [44]:
prompt = [
{"role": "system", "content": system_prompt},
{"role": "user", "content": user_prompt},
]

In [46]:
from schema import DailyReport

In [48]:
import time, httpx, json

# 비동기 HTTP 클라이언트 전역 생성 ( 재사용을 통해 커넥션 풀 활용 및 성능 최적화 )
client = httpx.AsyncClient(
    timeout=httpx.Timeout(
        connect=5.0,   # TCP 연결 수립 최대 대기 시간
        read=120.0,    # 응답 바디 수신 최대 대기 시간 (LLM 추론 고려)
        write=30.0,    # 요청 바디 전송 최대 시간
        pool=5.0       # 커넥션 풀에서 대기 가능한 최대 시간
    ),
    limits=httpx.Limits(
        max_connections=50,          # 동시에 열 수 있는 최대 TCP 연결 수
        max_keepalive_connections=20 # Keep-Alive 유지 연결 수
    ),
)

In [58]:
def clean_text(text: str) -> str:
    import re
    import unicodedata

    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u202f", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [59]:
payload = {
    "model": "gpt-oss-20b",
    "messages": prompt,
    "temperature": 0.0,
    "max_tokens": 4096,
    # LLM 출력이 반드시 CauseList 스키마를 따르도록 강제
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "cause-list",
            "schema": DailyReport.model_json_schema(),
        },
    },
}

resp = await client.post("http://10.122.100.173:8000/v1/chat/completions", json=payload)
resp.raise_for_status()

data = resp.json()


result = json.loads(clean_text(data["choices"][0]["message"]["content"]))

In [60]:
result["report_date"] = date_str

In [61]:
result

{'report_date': '2026-03-18',
 'overall_level': 'medium',
 'summary': '서비스는 대부분 정상 가동했으나 22:00:40~22:01:10 사이에 짧은 다운이 발생했고, 여러 지연 스파이크가 반복적으로 관측되었습니다.',
 'top_incident': {'time': '22:00:40',
  'level': 'high',
  'summary': '서비스 상태가 0으로 떨어져 요청 처리량이 0이 되고 지연이 15 ms 이상으로 급증',
  'impact': '서비스 다운으로 인한 요청 실패 및 지연 증가'},
 'patterns': [{'type': '지연 스파이크',
   'summary': 'p95/p99 지연이 10 ms~50 ms 범위에서 반복적으로 급증했으며, 대부분 서비스 가용성에는 영향이 없었음'}],
 'timeline_compact': [{'time': '14:42:10',
   'level': 'medium',
   'summary': 'latency_p95/p99가 32~86 ms로 급증'},
  {'time': '15:00:10',
   'level': 'medium',
   'summary': '메모리 압력 급증과 함께 지연이 60 ms까지 상승'},
  {'time': '15:33:10',
   'level': 'medium',
   'summary': 'latency_p95/p99가 12~13 ms로 급증'},
  {'time': '19:25:40',
   'level': 'medium',
   'summary': 'latency_p95/p99가 272~340 ms 급증, 처리량이 5배 증가'},
  {'time': '20:30:10',
   'level': 'medium',
   'summary': 'p95/p99 지연이 약 50 ms로 급증'},
  {'time': '21:36:10',
   'level': 'medium',
   'summary': 'p95/p99 지연이 21~2

In [ ]:
# from app.core.config import (
#     DAILY_REPORT_SYSTEM_PROMPT_PATH,
# )

# from utils import load_system_prompt

# system_prompt = load_system_prompt(INTERVAL_REPORT_SYSTEM_PROMPT_PATH)

# schema = 
# print(schema)

In [ ]:
base_dir = "./logs/report/interval"

date_str = start_time.strftime("%Y-%m-%d")
dir_path = os.path.join(base_dir, date_str)